In [ ]:
import json
import os
import random
import re
import time
import requests
from getpass import getpass
from pathlib import Path
from tqdm import tqdm

# 0. Tự dò thư mục gốc của repo để chạy local trên máy của bạn
def detect_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data").exists():
            return candidate
    return here

PROJECT_ROOT = detect_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. Cấu hình API FPT
def load_api_key() -> str:
    api_key = os.getenv("FPT_API_KEY")  
    if not api_key:
        try:
            api_key = getpass("Enter FPT API key: ").strip()
        except Exception:
            api_key = ""
    if not api_key:
        raise ValueError("Thiếu API key. ")
    return api_key

API_KEY = load_api_key()
MODEL_NAME_CANDIDATES = [
    "SaoLa-Llama3.1-planner",
    "saola-llama3.1-planner",
 ]
API_URL_CANDIDATES = [
    "https://mkp-api.fptcloud.com/v1/chat/completions",
    "https://mkp-api.fptcloud.com/chat/completions",
 ]

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "x-api-key": API_KEY,
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# 2. Cấu hình đường dẫn (local machine / VS Code / Jupyter)
INPUT_FILE = PROJECT_ROOT / "data/processed/week8_html_css_javascript_chunks.json"
OUTPUT_FILE = PROJECT_ROOT / "data/QA/week8_html_css_javascript.json"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy input file: {INPUT_FILE}. Hãy đảm bảo repo đã có data/processed/week0_scratch_chunks.json."
    )

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_FILE)
print("Output:", OUTPUT_FILE)

system_prompt = """You are an expert Computer Science teaching assistant for CS50x.
Based on the provided video chunk context (Transcript, OCR, Visual), generate exactly 2 complex questions.

Requirements:
1. Questions must require reasoning from BOTH the Transcript and Visual elements (OCR/Visual Description). Do not ask questions solvable by transcript alone.
2. Output language for questions and answers: English.
3. Return STRICTLY a raw JSON array matching this exact schema. Do not output markdown code blocks (like ```json) or any conversational text.

[
  {
    \"question_id\": \"<video_id>_<random_number>\",
    \"question\": \"Complex question in English\",
    \"target_answer\": \"Correct answer in English\",
    \"question_type\": \"code_explanation | visual_reasoning | concept_understanding\"
  }
]"""

def extract_json_array_text(text: str) -> str:
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```$", "", s)
    start = s.find("[")
    end = s.rfind("]")
    if start != -1 and end != -1 and end > start:
        return s[start:end + 1]
    return s

def _single_call(api_url: str, model_name: str, payload_messages: list, timeout: int = 60) -> str:
    payload = {
        "model": model_name,
        "messages": payload_messages,
        "temperature": 0.2,
        "max_tokens": 1024,
        "top_p": 1,
        "top_k": 40,
        "presence_penalty": 0,
        "frequency_penalty": 0,
        "stream": False,
    }
    resp = requests.post(api_url, headers=HEADERS, json=payload, timeout=timeout)
    if resp.status_code >= 400:
        preview = resp.text[:300]
        raise RuntimeError(f"HTTP {resp.status_code}: {preview}")
    body = resp.json()
    content = body.get("choices", [{}])[0].get("message", {}).get("content", "")
    if not content:
        raise RuntimeError(f"Response thiếu message.content: {body}")
    return content

def detect_working_route() -> tuple[str, str]:
    test_messages = [
        {"role": "user", "content": "Can you reply exactly: OK"},
    ]
    errors = []
    for api_url in API_URL_CANDIDATES:
        for model_name in MODEL_NAME_CANDIDATES:
            try:
                content = _single_call(api_url, model_name, test_messages, timeout=30)
                print(f"[OK] API route: {api_url} | model: {model_name} | sample: {content[:40]}")
                return api_url, model_name
            except Exception as exc:
                errors.append(f"{api_url} | {model_name} -> {exc}")
    raise RuntimeError("Không tìm được route/model hoạt động. Chi tiết:\n" + "\n".join(errors))

API_URL, MODEL_NAME = detect_working_route()
print("Using API_URL:", API_URL)
print("Using MODEL_NAME:", MODEL_NAME)

def call_fpt_chat(payload_messages: list, max_retries: int = 3, timeout: int = 120) -> str:
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return _single_call(API_URL, MODEL_NAME, payload_messages, timeout=timeout)
        except Exception as exc:
            last_error = exc
            if attempt < max_retries:
                time.sleep(1.5 * attempt)
            else:
                raise RuntimeError(f"Gọi API thất bại sau {max_retries} lần: {exc}") from exc
    raise RuntimeError(str(last_error))

# 3. Đọc dữ liệu
with INPUT_FILE.open("r", encoding="utf-8") as f:
    chunks = json.load(f)

qa_dataset = []
error_count = 0

# 4. Chạy vòng lặp gọi API
for chunk in tqdm(chunks):
    if not chunk.get("transcript") and not chunk.get("ocr_text"):
        continue

    context = (
        f"Transcript: {chunk.get('transcript', '')}\n"
        f"OCR: {chunk.get('ocr_text', '')}\n"
        f"Visual: {chunk.get('visual_description', '')}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context},
    ]

    try:
        result_text = call_fpt_chat(messages)
        qa_list = json.loads(extract_json_array_text(result_text))

        if not isinstance(qa_list, list):
            raise ValueError(f"Model không trả list JSON: {type(qa_list)}")

        for qa in qa_list:
            if not isinstance(qa, dict):
                continue
            qa.setdefault("question_id", f"{chunk.get('video_id', 'unknown')}_{random.randint(1000, 9999)}")
            qa.setdefault("question", "")
            qa.setdefault("target_answer", "")
            qa.setdefault("question_type", "concept_understanding")
            qa["ground_truth_video_id"] = chunk.get("video_id", "")
            qa["ground_truth_window"] = [chunk.get("start_time", 0.0), chunk.get("end_time", 0.0)]
            qa_dataset.append(qa)
    except Exception as e:
        error_count += 1
        print(f"Lỗi chunk {chunk.get('chunk_id', 'unknown')}: {e}")

# 5. Lưu kết quả
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=4)

print("Saved:", OUTPUT_FILE)
print("Total QA:", len(qa_dataset))
print("Total errors:", error_count)

PROJECT_ROOT: C:\github\Multimodal-Tutor
Input: C:\github\Multimodal-Tutor\data\processed\week8_html_css_javascript_chunks.json
Output: C:\github\Multimodal-Tutor\data\QA\week8_html_css_javascript.json
[OK] API route: https://mkp-api.fptcloud.com/v1/chat/completions | model: SaoLa-Llama3.1-planner | sample: OK
Using API_URL: https://mkp-api.fptcloud.com/v1/chat/completions
Using MODEL_NAME: SaoLa-Llama3.1-planner


 59%|█████▉    | 94/159 [02:40<01:52,  1.73s/it]

Lỗi chunk cs50_week8_html_css_javascript_094: Invalid \escape: line 4 column 72 (char 117)


 60%|██████    | 96/159 [02:42<01:38,  1.56s/it]

Lỗi chunk cs50_week8_html_css_javascript_096: Invalid \escape: line 5 column 41 (char 263)


 84%|████████▍ | 134/159 [03:55<00:50,  2.03s/it]

Lỗi chunk cs50_week8_html_css_javascript_134: Invalid control character at: line 11 column 166 (char 1084)


100%|██████████| 159/159 [04:41<00:00,  1.77s/it]

Saved: C:\github\Multimodal-Tutor\data\QA\week8_html_css_javascript.json
Total QA: 305
Total errors: 3
